# EuroSAT Multispectral Classification Model

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import rasterio

class EuroSATMultispectral(Dataset):
    def __init__(self, csv_path, root_dir):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rel_path = row['filename'].replace('\\', os.sep).replace('.jpg', '.tif')
        full_path = os.path.join(self.root_dir, rel_path)

        with rasterio.open(full_path) as src:
            img = src.read().astype(np.float32) / 5000.0

        return torch.tensor(img), int(row['label'])

In [ ]:
import torch
import torch.nn as nn
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
#change this according to your path
train_ds = EuroSATMultispectral(
    r"train.csv",   #add the path of csv file for training data and training images folder
    r"train"
)
val_ds = EuroSATMultispectral(
    r"validation.csv",  #add the path of csv file for validation data and validation images folder
    r"val"
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

# sanity check — run this before anything else
imgs, labels = next(iter(train_loader))
print(imgs.shape)    # should be (32, 13, 64, 64)
print(labels.shape)  # should be (32,)

In [ ]:
model = timm.create_model('resnet18', pretrained=True, num_classes=10)

old_conv = model.conv1
new_conv = nn.Conv2d(13, 64, kernel_size=7, stride=2, padding=3, bias=False)

with torch.no_grad():
    new_conv.weight = nn.Parameter(
        old_conv.weight.mean(dim=1, keepdim=True).repeat(1, 13, 1, 1)
    )

model.conv1 = new_conv
model = model.to(device)
print("model ready")

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
best_val_acc = 0.0

for epoch in range(20):
    # --- train ---
    model.train()
    total_loss, correct = 0.0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = total_loss / len(train_loader)
    train_acc = correct / len(train_ds)

    # --- validate ---
    model.eval()
    val_correct = 0
    val_loss = 0.0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = val_correct / len(val_ds)

    print(
        f"Epoch {epoch+1}: "
        f"train_loss={train_loss:.4f} "
        f"train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.4f} "
        f"val_acc={val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "ms_model.pth")
        print("Saved best model.")

In [ ]:
class_names = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake"
]

In [16]:
class EuroSATTestSet(Dataset):
    def __init__(self, test_dir):
        self.test_dir = test_dir
        self.filenames = sorted(os.listdir(test_dir))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        path = os.path.join(self.test_dir, fname)
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32) / 5000.0
        return torch.tensor(img), fname

test_ds = EuroSATTestSet(r"EuroSATallBands_test_flat") #add the path of test 
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

idx_to_class = {i: name for i, name in enumerate(class_names)}

results = []
model.eval()
with torch.no_grad():
    for imgs, fnames in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        for fname, pred in zip(fnames, preds):
            results.append({"img_id": fname, "predicted_label": idx_to_class[pred]})

pd.DataFrame(results).to_csv("ms_predictions.csv", index=False)